# CymbalGoal AlloyDB Workshop—Index Diagnostics: why the 03 numbers look the way they do

**Track A, pipeline step 04.** Notebook 03 found HNSW about 2.6 times ScaNN's throughput on plain
searches, ScaNN about 2.4 times HNSW's on filtered ones, and two oddities. This notebook explains them,
using the database 03 left behind (`cymbalgoal_bench`: no reload, no rebuild).

| Section | Question |
| :-- | :-- |
| **ScaNN tuning** | ScaNN tops out near 290 searches a second. Is it the re-ranking step (`scann.pre_reordering_num_neighbors`), the two search threads each query uses (`scann.num_search_threads`), or the number of leaves searched? |
| **HNSW and the planner** | Why did a search jump from 11 ms to 2,177 ms at `hnsw.ef_search = 250`? The estimated cost of each plan is shown next to the plan the planner chose. |
| **Plans** | `EXPLAIN ANALYZE` for every search shape: which index, how many buffers, and for filtered searches whether a vector index is used at all. |
| **Filtered strategies** | Five ways to run "one season, `SIGN` only": no vector index, each index as the planner chooses it, and each index forced, with and without `hnsw.iterative_scan`. |
| **Load** | 64 concurrent searches per configuration, from **four client processes** instead of one, plus one single-process control. Each step records what the database saw (calls and execution time from `pg_stat_statements`), and the end shows the instance's CPU, minute by minute, from Cloud Monitoring. |

**Second run, on a cluster with the columnar engine off.** The first run found that once AlloyDB's
columnar engine had loaded these tables, the planner chose a full columnar scan over either vector
index. The Terraform now leaves the columnar engine off, so this notebook doesn't touch it. Every
tuning and load measurement forces the vector index; the plans also show what the planner picks on its
own. The Cloud Monitoring API is enabled at the start so the CPU timeline works.

Runs in about 25 minutes. Run all; the last cell is the handback.

---
## 1 · Configuration

In [ ]:
TARGET_RECALL     = 0.95
SCANN_BASE_LEAVES = 39                         # 03's choice: 20% of ~196 leaves, 99% recall
SCANN_LEAVES      = [20, 29, 39, 59]           # scann.num_leaves_to_search
SCANN_REORDER     = [None, 200, 100, 50, 20, 0]  # scann.pre_reordering_num_neighbors; None = the default
SCANN_THREADS     = [1, 2]                     # scann.num_search_threads
HNSW_EF           = [100, 160, 200, 225, 250, 300, 400]
HNSW_MATCH_SCANN  = True                       # HNSW gets the smallest ef_search that matches ScaNN tuned's recall
SCANN_FILTER_LEAVES  = [29, 59, 98]            # filtered searches need more of the index searched
SCANN_FILTER_REORDER = [100, 200, None]
RUN_LOAD          = True
LOAD_CLIENTS      = 64
CLIENT_PROCS      = 4                          # client processes for the load steps
STEP_SECONDS      = 60                         # measured window per load step, after a 5-second warm-up

In [ ]:
import os, sys, subprocess

if not os.environ.get("CG_FIXTURE"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "google-cloud-alloydb-connector[pg8000]", "pandas"], check=True)
print("client libraries ready")

In [ ]:
# ---- Where things live ---------------------------------------------------------------------
GCS_READ       = "gs://class-demo/alloydb-labs/cymbalgoal"                       # read-only
GCS_WORK       = "gs://class-demo/alloydb-labs/cymbalgoal-alloydb-workshop"
RESULTS_PREFIX = f"{GCS_WORK}/benchmarks"                                       # the only course-bucket write
DB_NAME        = "cymbalgoal_bench"                                             # built by 03

---
## 2 · Environment and guard

In [ ]:
import csv, gzip, io, json, math, re, threading, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

_FIXTURE = os.environ.get("CG_FIXTURE")          # offline tests only; unset in Colab


def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


BASE = Path("/content/cymbalgoal-loadtest")
try:
    BASE.mkdir(parents=True, exist_ok=True)
except OSError:
    BASE = Path.home() / "cymbalgoal-loadtest"
    BASE.mkdir(parents=True, exist_ok=True)

if _FIXTURE:
    PROJECT, USER, REGION = "fixture-project", "postgres", "us-central1"
    CLUSTER, INSTANCE, PROJECT_NUMBER = "cymbalgoal-cluster", "cymbalgoal-primary", "123"
    SHAPE = {"cpu": 8, "availability": "ZONAL", "version": "POSTGRES_18"}
else:
    PROJECT = sh("gcloud config get-value project")
    USER = sh("gcloud config get-value account")
    PROJECT_NUMBER = sh(f"gcloud projects describe {PROJECT} --format='value(projectNumber)'")
    clusters = json.loads(sh("gcloud alloydb clusters list --format=json") or "[]")
    assert clusters, "No AlloyDB cluster in this project. Apply the Terraform first."
    _cl = clusters[0]
    CLUSTER = _cl["name"].split("/")[-1]
    REGION = _cl["name"].split("/locations/")[1].split("/")[0]
    _inst = [i for i in json.loads(sh(f"gcloud alloydb instances list --cluster={CLUSTER} "
                                      f"--region={REGION} --format=json") or "[]")
             if i.get("instanceType") == "PRIMARY"]
    assert _inst, f"No PRIMARY instance in cluster {CLUSTER}."
    INSTANCE = _inst[0]["name"].split("/")[-1]
    SHAPE = {"cpu": _inst[0].get("machineConfig", {}).get("cpuCount"),
             "availability": _inst[0].get("availabilityType"),
             "version": _cl.get("databaseVersion")}

INSTANCE_URI = f"projects/{PROJECT}/locations/{REGION}/clusters/{CLUSTER}/instances/{INSTANCE}"
PROJECT_BUCKET = f"gs://{PROJECT}-cymbalgoal-staging"
SERVICE_AGENT = f"service-{PROJECT_NUMBER}@gcp-sa-alloydb.iam.gserviceaccount.com"

# ---- Guarded Cloud Storage -----------------------------------------------------------------
GCS_LOG = []


def _under(uri, prefix):
    p = prefix.rstrip("/")
    return uri == p or uri.startswith(p + "/")


READ_OK = (GCS_READ, GCS_WORK, PROJECT_BUCKET)
WRITE_OK = (RESULTS_PREFIX, PROJECT_BUCKET)


def _guard(args):
    verb = args[0]
    uris = [a for a in args[1:] if a.startswith("gs://")]
    if verb in ("ls", "cat"):
        bad = [u for u in uris if not any(_under(u, p) for p in READ_OK)]
    elif verb == "cp":
        if not any(_under(args[-1], p) for p in WRITE_OK):
            raise PermissionError(f"gcs(): writes only under {WRITE_OK}, refused {args[-1]}")
        bad = [u for u in uris[:-1] if not any(_under(u, p) for p in READ_OK)]
    else:
        raise PermissionError(f"gcs(): '{verb}' refused—only ls, cat and cp are allowed")
    if bad:
        raise PermissionError(f"gcs(): outside the allowed prefixes: {bad}")
    GCS_LOG.append("gcloud storage " + " ".join(args))


def _fx(uri):
    return Path(_FIXTURE) / uri[len("gs://"):]


def gcs(*args):
    _guard(args)
    if _FIXTURE:
        import shutil
        if args[0] == "cat":
            p = _fx(args[-1])
            return subprocess.CompletedProcess(args, 0 if p.exists() else 1,
                                               p.read_text() if p.exists() else "", "" if p.exists() else "No URLs matched")
        if args[0] == "ls":
            p = _fx(args[-1])
            return subprocess.CompletedProcess(args, 0 if p.exists() else 1, args[-1] if p.exists() else "", "")
        d = _fx(args[-1]); d.parent.mkdir(parents=True, exist_ok=True)
        src = args[-2]
        shutil.copy(_fx(src) if src.startswith("gs://") else src, d)
        return subprocess.CompletedProcess(args, 0, "", "")
    return subprocess.run(["gcloud", "storage", *args], capture_output=True, text=True)


def gcs_stream(uri):
    _guard(("cat", uri))
    if _FIXTURE:
        fh = open(_fx(uri), "rb")
        return fh, fh.close
    p = subprocess.Popen(["gcloud", "storage", "cat", uri], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)

    def _close():
        p.stdout.close(); p.kill(); p.wait()
    return p.stdout, _close


def get_text(uri):
    r = gcs("cat", uri)
    return r.stdout if r.returncode == 0 else None


# ---- Results and handback ------------------------------------------------------------------
HB, RESULTS = {}, {}


def hb(section, line=""):
    HB.setdefault(section, []).append(str(line))


def mins(s):
    if s is None:
        return "—"
    return f"{s:.0f} s" if s < 90 else f"{s / 60:.1f} min"


print(f"project   {PROJECT}  (#{PROJECT_NUMBER})")
print(f"cluster   {CLUSTER}  {REGION}  {SHAPE}")
print(f"instance  {INSTANCE}")
print(f"you       {USER}")
print(f"runtime   {os.cpu_count()} vCPU")
hb("2 · Environment", f"project {PROJECT}; cluster {CLUSTER} {REGION}; instance {INSTANCE} {SHAPE}; "
                      f"you {USER}; runtime {os.cpu_count()} vCPU")

---
## 3 · Connect

In [ ]:
if _FIXTURE:
    import pg8000.dbapi

    def _raw_connect(db):
        return pg8000.dbapi.connect(user="postgres", host=os.environ.get("CG_PGHOST", "localhost"),
                                    port=int(os.environ.get("CG_PGPORT", "5432")), database=db)
else:
    from google.cloud.alloydb.connector import Connector, IPTypes
    connector = Connector()

    def _raw_connect(db):
        return connector.connect(INSTANCE_URI, "pg8000", user=USER, db=db,
                                 enable_iam_auth=True, ip_type=IPTypes.PUBLIC)


def connect(db=DB_NAME):
    c = _raw_connect(db)
    c.autocommit = True
    return c


_tls = threading.local()


def session():
    c = getattr(_tls, "conn", None)
    if c is not None:
        try:
            cur = c.cursor(); cur.execute("SELECT 1"); cur.close()
            return c
        except Exception:
            pass
    _tls.conn = connect()
    return _tls.conn


def _drain(conn, show=False):
    out = []
    for n in list(getattr(conn, "notices", None) or []):
        m = n.get(b"M") if isinstance(n, dict) else None
        if m:
            out.append(m.decode())
    try:
        conn.notices.clear()
    except Exception:
        pass
    if show:
        for m in out:
            print("   ", m)
    return out


def run(sql, params=None, show_notices=False):
    conn = session()
    cur = conn.cursor()
    cur.execute(sql, params) if params else cur.execute(sql)
    cur.close()
    return _drain(conn, show_notices)


def q(sql, params=None):
    conn = session()
    cur = conn.cursor()
    cur.execute(sql, params) if params else cur.execute(sql)
    cols = [d[0] for d in cur.description] if cur.description else []
    rows = cur.fetchall() if cur.description else []
    cur.close()
    _drain(conn)
    return pd.DataFrame(rows, columns=cols)


t0 = time.time()
_c = connect("postgres")
_cur = _c.cursor()
_cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DB_NAME,))
if _cur.fetchone() is None:
    _cur.execute(f'CREATE DATABASE "{DB_NAME}"')
    print(f"created database {DB_NAME}")
else:
    print(f"database {DB_NAME} already exists")
_cur.execute("SELECT version()")
print(_cur.fetchone()[0].split(" on ")[0], f"— connected in {time.time() - t0:.1f}s")
_cur.close(); _c.close()

EXTS = ["vector"] if _FIXTURE else ["vector", "alloydb_scann", "google_ml_integration"]
for ext in EXTS:
    run(f"CREATE EXTENSION IF NOT EXISTS {ext}")
if _FIXTURE:   # offline stand-in for the in-database model call: a fixed pseudo-embedding per text
    run("CREATE SCHEMA IF NOT EXISTS ai")
    run('''CREATE OR REPLACE FUNCTION ai.embedding(model text, content text) RETURNS vector
           LANGUAGE sql IMMUTABLE AS $$
             SELECT array_agg(sin(i * (abs(hashtext(content)) % 997 + 1)))::vector
             FROM generate_series(1, 3072) AS i $$''')
display(q("SELECT extname, extversion FROM pg_extension ORDER BY 1"))

---
## 4 · What 03 left behind

Checks that 03's tables and indexes are here, lists every ScaNN and HNSW setting with its current
value, and records sizes, including the TOAST storage that holds the full-precision vectors (each one is
12 KB, too big to sit in the table row). The exact top ten for each search is computed once and kept in
`qexact`, so a rerun skips it.

In [ ]:
S = "4 · Setup"
NEED = ["reports", "qvec", "bench_scann", "bench_hnsw", "bench_scann_idx", "bench_hnsw_idx"]
missing = [n for n in NEED if not q("SELECT to_regclass(%s) IS NOT NULL AS x", (n,))["x"][0]]
assert not missing, (f"missing {missing}. Run 03 first: RUN_LOAD = False is enough (corpus, queries, "
                     f"indexes and tuning, about 6 minutes).")
N_ROWS = int(q("SELECT count(*) AS n FROM reports")["n"][0])
VER = q("SELECT extname, extversion FROM pg_extension WHERE extname IN ('vector', 'alloydb_scann')")
VER = dict(zip(VER.extname, VER.extversion))

IDX = {
    "ScaNN": {"table": "bench_scann", "index": "bench_scann_idx", "col": "report_embedding", "qcol": "v"},
    "HNSW":  {"table": "bench_hnsw", "index": "bench_hnsw_idx",
              "col": "report_embedding::halfvec(3072)", "qcol": "vh"},
    # full precision on bench_hnsw: the HNSW index is on the halfvec expression, so it can't serve this
    "Exact": {"table": "bench_hnsw", "index": None, "col": "report_embedding", "qcol": "v"},
}
if _FIXTURE:
    IDX["HNSW"].update(col="report_embedding", qcol="v")
FILTER = "season = (SELECT season FROM qvec WHERE id = %s) AND recommendation = 'SIGN'"
FORCE = [("enable_seqscan", "off"), ("enable_bitmapscan", "off"), ("enable_sort", "off"), ("jit", "off")]
ITER = ("hnsw.iterative_scan", "relaxed_order")


def search_sql(name, filtered):
    ix = IDX[name]
    where = f"WHERE {FILTER} " if filtered else ""
    return (f"SELECT report_id FROM {ix['table']} {where}"
            f"ORDER BY {ix['col']} <=> (SELECT {ix['qcol']} FROM qvec WHERE id = %s) LIMIT 10")


def args_for(filtered, qid):
    return (qid, qid) if filtered else (qid,)


# one search on each table loads both extensions into this session, so SHOW can see their settings
for name in ("ScaNN", "HNSW"):
    q(search_sql(name, False), (0,))
KNOB = {}
for k in ["scann.num_leaves_to_search", "scann.pre_reordering_num_neighbors", "scann.num_search_threads",
          "hnsw.ef_search", "hnsw.iterative_scan", "hnsw.max_scan_tuples"]:
    try:
        KNOB[k] = str(q(f"SHOW {k}").iloc[0, 0])
    except Exception:  # noqa: BLE001
        KNOB[k] = None
ALLG = q("SELECT name, setting, unit FROM pg_settings WHERE name ~ '^(scann|hnsw)[.]' "
         "OR name IN ('shared_buffers', 'work_mem', 'max_parallel_workers_per_gather', 'max_connections', "
         "'jit', 'jit_above_cost', 'random_page_cost', 'effective_cache_size') "
         "ORDER BY name")
GUCS = [f"{r.name} = {r.setting}{' ' + r.unit if isinstance(r.unit, str) else ''}" for r in ALLG.itertuples()]


def clean(sets):
    # drops 'default' entries and any setting this server doesn't have
    return [(k, v) for k, v in sets
            if v is not None and (k.startswith("enable_") or k == "jit" or KNOB.get(k) is not None)]


SIZES = {}
for t in ("bench_scann", "bench_hnsw", "bench_scann_idx", "bench_hnsw_idx"):
    SIZES[t] = int(q("SELECT pg_relation_size(%s) AS b", (t,))["b"][0]) / 1e6
for t in ("bench_scann", "bench_hnsw"):
    SIZES[t + " incl. TOAST"] = int(q("SELECT pg_table_size(%s) AS b", (t,))["b"][0]) / 1e6
OPTS = {r.relname: r.reloptions for r in q("SELECT relname, reloptions FROM pg_class "
                                              "WHERE relname IN ('bench_scann_idx', 'bench_hnsw_idx')").itertuples()}

MON = "skipped offline"
if not _FIXTURE:   # Cloud Monitoring is read at the end, for the CPU timeline
    _r = subprocess.run(["gcloud", "services", "enable", "monitoring.googleapis.com", f"--project={PROJECT}"],
                        capture_output=True, text=True)
    MON = "enabled" if _r.returncode == 0 else f"enable failed: {_r.stderr.strip()[-200:]}"

try:
    run("CREATE EXTENSION IF NOT EXISTS pg_stat_statements")
    q("SELECT calls FROM pg_stat_statements LIMIT 1")
    PSS = True
except Exception as e:  # noqa: BLE001
    PSS = False
    print(f"pg_stat_statements unavailable ({str(e)[:120]}): the load steps won't show server-side time")

# ---- exact answers, kept in qexact ---------------------------------------------------------
QV = q("SELECT id, season FROM qvec ORDER BY id")
QIDS = [int(x) for x in QV.id]
SEASON = dict(zip(QIDS, (int(s) for s in QV.season)))
run("CREATE TABLE IF NOT EXISTS qexact (id int, filtered bool, ids text, PRIMARY KEY (id, filtered))")
t0 = time.time()
if int(q("SELECT count(*) AS n FROM qexact")["n"][0]) != 2 * len(QIDS):
    run("TRUNCATE qexact")

    def _exact(job):
        qid, filtered = job
        run("SET enable_indexscan = off")
        where = f"WHERE season = {SEASON[qid]} AND recommendation = 'SIGN' " if filtered else ""
        ids = list(q(f"SELECT report_id FROM reports {where}"
                     f"ORDER BY report_embedding <=> (SELECT v FROM qvec WHERE id = {qid}) LIMIT 10").report_id)
        run("INSERT INTO qexact VALUES (%s, %s, %s)", (qid, filtered, json.dumps([str(i) for i in ids])))

    with ThreadPoolExecutor(max_workers=8) as pool:
        list(pool.map(_exact, [(i, f) for i in QIDS for f in (False, True)]))
    print(f"exact answers computed in {time.time() - t0:.0f}s")
EX = q("SELECT id, filtered, ids FROM qexact")
TRUTH = {(int(r.id), bool(r.filtered)): set(json.loads(r.ids)) for r in EX.itertuples()}

print(f"{N_ROWS:,} reports; extensions {VER}; pg_stat_statements: {PSS}")
print("settings:", *GUCS, sep="\n  ")
print("sizes MB:", {k: round(v) for k, v in SIZES.items()})
print("index options:", OPTS)
print(f"Cloud Monitoring API: {MON}")
hb(S, f"{N_ROWS:,} rows; extensions {VER}; pg_stat_statements {PSS}; client runtime {os.cpu_count()} vCPU")
hb(S, "settings: " + "; ".join(GUCS))
hb(S, "sizes MB: " + ", ".join(f"{k} {v:,.0f}" for k, v in SIZES.items()))
hb(S, f"index options: {OPTS}")
hb(S, f"Cloud Monitoring API: {MON}")

In [ ]:
# ---- shared helpers: settings, recall, plans -----------------------------------------------
def apply(sets):
    done = []
    try:
        for k, v in clean(sets):
            run(f"SET {k} = {v}")
            done.append(k)
    except Exception:
        reset(done)
        raise
    return done


def reset(keys):
    for k in keys:
        run(f"RESET {k}")


def measure(name, filtered, sets=(), n=None):
    keys = apply(sets)
    try:
        sql, rec, rows, ms = search_sql(name, filtered), [], [], []
        for qid in (QIDS if n is None else QIDS[:n]):
            t0 = time.time()
            ids = [str(x) for x in q(sql, args_for(filtered, qid)).report_id]
            ms.append((time.time() - t0) * 1000)
            truth = TRUTH[(qid, filtered)]
            rec.append(len(set(ids) & truth) / max(1, len(truth)))
            rows.append(len(ids))
    finally:
        reset(keys)
    ms.sort()
    return {"recall": sum(rec) / len(rec), "rows": sum(rows) / len(rows),
            "p50": ms[len(ms) // 2], "mean": sum(ms) / len(ms), "n": len(ms)}


_ACCESS = re.compile(r"(Custom Scan \(columnar scan\) on \S+|Seq Scan on \S+|Index Scan using \S+|Index Only Scan using \S+|Bitmap Index Scan on \S+)")


def explain(name, filtered, sets=(), analyze=True, qid=0):
    keys = apply(sets)
    try:
        sql = search_sql(name, filtered).replace("%s", str(qid))
        opts = "(ANALYZE, BUFFERS)" if analyze else ""
        text = "\n".join(q(f"EXPLAIN {opts} {sql}").iloc[:, 0])
    finally:
        reset(keys)
    access = [a for a in dict.fromkeys(_ACCESS.findall(text)) if "qvec" not in a]
    m_cost = re.search(r"cost=[\d.]+\.\.([\d.]+)", text)
    m_exec = re.search(r"Execution Time: ([\d.]+)", text)
    m_buf = re.search(r"Buffers: shared ([^\n]+)", text)
    return {"access": " + ".join(access) + (" + Sort" if re.search(r"\bSort\b", text) else ""),
            "cost": float(m_cost.group(1)) if m_cost else None,
            "exec_ms": float(m_exec.group(1)) if m_exec else None,
            "buffers": m_buf.group(1).strip() if m_buf else "", "text": text,
            "vector_index": IDX[name]["index"] is not None and IDX[name]["index"] in text}


def bufcount(b):
    return sum(int(x) for x in re.findall(r"=(\d+)", b or ""))

---
## 5 · ScaNN: re-ranking, leaves, threads

ScaNN finds candidates using compressed (8-bit) vectors, then **re-ranks** the best of them using the
full vectors, which live in TOAST storage. By default it re-ranks 50 candidates per row asked for: 500
for a top ten. Each search here varies three settings:

* `scann.num_leaves_to_search`: how much of the index is searched
* `scann.pre_reordering_num_neighbors`: how many candidates are re-ranked (0 turns re-ranking off)
* `scann.num_search_threads`: threads per search, which cut one search's latency but not the total work

Every measurement forces the index (sequential and bitmap scans and sorts are switched off); the
**planner picks** column shows what the planner would have chosen on its own. **Buffers** is the number
of 8 KB pages one search touches; most of them are re-ranking reads. The **tuned** setting is the fastest single-thread combination that still reaches the target recall.

In [ ]:
S = "5 · ScaNN"
SCANN_BASE = [("scann.num_leaves_to_search", SCANN_BASE_LEAVES)]
threads = SCANN_THREADS if KNOB.get("scann.num_search_threads") is not None else [None]
reorders = SCANN_REORDER if KNOB.get("scann.pre_reordering_num_neighbors") is not None else [None]
leaves = SCANN_LEAVES if KNOB.get("scann.num_leaves_to_search") is not None else [None]
print(f"defaults: { {k: v for k, v in KNOB.items() if k.startswith('scann')} }")
hb(S, f"defaults: { {k: v for k, v in KNOB.items() if k.startswith('scann')} }")
SC = []
t0 = time.time()
for lv in leaves:
    for ro in reorders:
        row = {"leaves": lv, "reorder": "default" if ro is None else ro}
        for th in threads:
            sets = [("scann.num_leaves_to_search", lv), ("scann.pre_reordering_num_neighbors", ro),
                    ("scann.num_search_threads", th)]
            try:
                m = measure("ScaNN", False, sets + FORCE)
            except Exception as e:  # noqa: BLE001
                row[f"err{th}"] = str(e)[:120]
                continue
            row["recall"] = m["recall"]
            row[f"p50 {th or 'default'} thr"] = m["p50"]
            if th == threads[0]:
                row["buffers"] = bufcount(explain("ScaNN", False, sets + FORCE)["buffers"])
                row["planner picks"] = explain("ScaNN", False, sets, analyze=False)["access"]
        SC.append(row)
SCDF = pd.DataFrame(SC)
print(SCDF.to_string(index=False, float_format=lambda x: f"{x:.3f}" if x < 1 else f"{x:.1f}"))
print(f"\n({time.time() - t0:.0f}s)")
for r in SC:
    hb(S, "; ".join(f"{k} {v:.1%}" if k == "recall" else f"{k} {v:.1f}" if isinstance(v, float) else f"{k} {v}"
                    for k, v in r.items()))

th1 = threads[0]
key = f"p50 {th1 or 'default'} thr"
ok_rows = [r for r in SC if r.get("recall", 0) >= TARGET_RECALL and key in r]
best = min(ok_rows, key=lambda r: r[key]) if ok_rows else None
if best:
    SCANN_TUNED = [("scann.num_leaves_to_search", best["leaves"]),
                   ("scann.pre_reordering_num_neighbors", None if best["reorder"] == "default" else best["reorder"]),
                   ("scann.num_search_threads", th1)]
else:
    SCANN_TUNED = SCANN_BASE + [("scann.num_search_threads", th1)]
SCANN_TUNED_RECALL = best["recall"] if best else None
line = (f"tuned: {clean(SCANN_TUNED)}" + (f"—recall {best['recall']:.1%}, p50 {best[key]:.1f} ms" if best else
                                          "—nothing reached the target; using the baseline"))
print(line)
hb(S, line)

---
## 6 · HNSW: `ef_search` and the planner

For each `hnsw.ef_search`: the estimated cost of the **index** (forced: sequential and bitmap scans,
sorts and JIT switched off), and the plan the **planner chooses** on its own. The index's estimated cost rises with `ef_search`; a full scan's estimate doesn't, because it
ignores the cost of reading and comparing 38,445 vectors of 12 KB each. Recall and latency come from the
forced plan; where the planner picks something else, its latency is measured on 4 searches.
`ef_search` is matched to ScaNN tuned's recall, so the load test compares like with like.

In [ ]:
S = "6 · HNSW"
HN = []
t0 = time.time()
for ef in HNSW_EF:
    sets = [("hnsw.ef_search", ef)]
    nat = explain("HNSW", False, sets, analyze=False)
    frc = explain("HNSW", False, sets + FORCE, analyze=False)
    m = measure("HNSW", False, sets + FORCE)
    row = {"ef_search": ef, "planner picks": nat["access"], "est. cost (planner)": nat["cost"],
           "est. cost (index)": frc["cost"], "recall": m["recall"], "p50 index": m["p50"]}
    row["p50 planner"] = m["p50"] if nat["vector_index"] else measure("HNSW", False, sets, n=4)["p50"]
    HN.append(row)
    line = (f"ef_search {ef:>4}: index cost {frc['cost']:,.0f}; recall {m['recall']:.1%}; index p50 {m['p50']:.1f} ms"
            f" | planner: {nat['access'] or '?'} (cost {nat['cost']:,.0f}, p50 {row['p50 planner']:.1f} ms)")
    print(line)
    hb(S, line)
HNDF = pd.DataFrame(HN)
goal = TARGET_RECALL
if HNSW_MATCH_SCANN and SCANN_TUNED_RECALL:
    goal = max(goal, SCANN_TUNED_RECALL - 0.005)
ok = [r for r in HN if r["recall"] >= goal]
HNSW_EF_CHOSEN = ok[0]["ef_search"] if ok else HNSW_EF[-1]
cliff = next((r["ef_search"] for r in HN if "bench_hnsw_idx" not in r["planner picks"]), None)
HNSW_SET = [("hnsw.ef_search", HNSW_EF_CHOSEN)]
line = (f"chosen ef_search {HNSW_EF_CHOSEN} (recall goal {goal:.1%}); the planner skips the index from ef_search "
        f"{cliff or '— (never)'}  "
        f"({time.time() - t0:.0f}s)")
print(line)
hb(S, line)

---
## 7 · Plans, and the ways to run a filtered search

First, ScaNN gets its own setting for filtered searches: when the filter discards most candidates, it
has to search more of the index to find ten that qualify. Then `EXPLAIN (ANALYZE, BUFFERS)` for each
search shape, with recall and latency over the 96 searches (4 searches where the plan is a full scan).
A filter keeps about 1.5% of the corpus (one season, `SIGN` only), so comparing every matching row
exactly, with no vector index at all, is a legitimate plan: the **exact** rows show what it costs.

In [ ]:
S = "7 · Plans"
# ---- ScaNN on filtered searches needs its own setting -------------------------------------
FSC = []
for lv in (SCANN_FILTER_LEAVES if KNOB.get("scann.num_leaves_to_search") is not None else [None]):
    for ro in (SCANN_FILTER_REORDER if KNOB.get("scann.pre_reordering_num_neighbors") is not None else [None]):
        sets = [("scann.num_leaves_to_search", lv), ("scann.pre_reordering_num_neighbors", ro),
                ("scann.num_search_threads", threads[0])]
        try:
            m = measure("ScaNN", True, sets + FORCE)
        except Exception as e:  # noqa: BLE001
            hb(S, f"ScaNN filtered leaves {lv} reorder {ro}: FAILED {str(e)[:150]}")
            continue
        FSC.append({"sets": sets, **m})
        line = (f"ScaNN filtered, forced: leaves {lv}, reorder {'default' if ro is None else ro}: "
                f"recall {m['recall']:.1%}, rows {m['rows']:.1f}, p50 {m['p50']:.1f} ms")
        print(line)
        hb(S, line)
okf = [r for r in FSC if r["recall"] >= TARGET_RECALL]
pick = min(okf, key=lambda r: r["p50"]) if okf else (max(FSC, key=lambda r: r["recall"]) if FSC else None)
SCANN_FTUNED = pick["sets"] if pick else SCANN_TUNED
line = f"ScaNN filtered setting: {clean(SCANN_FTUNED)}" + (
    f"—recall {pick['recall']:.1%}, p50 {pick['p50']:.1f} ms" if pick else "")
print(line + "\n")
hb(S, line)

E = f"ef_search {HNSW_EF_CHOSEN}"
SHAPES = [
    ("plain: ScaNN 03 settings, planner's choice", "ScaNN", False, SCANN_BASE),
    ("plain: ScaNN 03 settings, index forced", "ScaNN", False, SCANN_BASE + FORCE),
    ("plain: ScaNN tuned, planner's choice", "ScaNN", False, SCANN_TUNED),
    (f"plain: HNSW {E}, planner's choice", "HNSW", False, HNSW_SET),
    (f"plain: HNSW {E}, index forced", "HNSW", False, HNSW_SET + FORCE),
    ("filtered: exact, planner's choice", "Exact", True, []),
    ("filtered: ScaNN tuned (plain setting), index forced", "ScaNN", True, SCANN_TUNED + FORCE),
    ("filtered: ScaNN filtered setting, index forced", "ScaNN", True, SCANN_FTUNED + FORCE),
    (f"filtered: HNSW {E}, planner's choice", "HNSW", True, HNSW_SET),
    (f"filtered: HNSW {E}, index forced", "HNSW", True, HNSW_SET + FORCE),
    (f"filtered: HNSW {E}, index forced, iterative scan", "HNSW", True, HNSW_SET + [ITER] + FORCE),
]
PLANS = {}
for label, name, filtered, sets in SHAPES:
    if label.endswith("iterative scan") and KNOB.get("hnsw.iterative_scan") is None:
        continue
    try:
        p = explain(name, filtered, sets)
        m = measure(name, filtered, sets, n=None if p["vector_index"] or filtered else 4)
    except Exception as e:  # noqa: BLE001
        print(f"{label}: FAILED {str(e)[:200]}")
        hb(S, f"{label}: FAILED {str(e)[:200]}")
        continue
    PLANS[label] = {**{k: v for k, v in p.items()}, **m}
    line = (f"{label}: {p['access']}; recall {m['recall']:.1%}; rows {m['rows']:.1f}; p50 {m['p50']:.1f} ms "
            f"(n={m['n']}); buffers {bufcount(p['buffers']):,}")
    print(line)
    hb(S, line)
print()
for label, p in PLANS.items():
    print(f"--- {label}\n{p['text']}\n")
    hb("7b · Plan text", f"--- {label}")
    for ln in p["text"].splitlines():
        hb("7b · Plan text", "    " + ln)

---
## 8 · Under load, from several client processes

Each step runs `LOAD_CLIENTS` connections spread over `CLIENT_PROCS` separate Python processes, so the
notebook can't be the bottleneck (03 ran every client as a thread in this one process). One step repeats
HNSW from a **single** process: if that one is slower, 03's HNSW numbers were limited by the client.

Besides queries per second and latency, each step reads `pg_stat_statements` before and after:

* **server ms / search**: the time the database spent executing each search
* **busy in server**: how many of the 64 searches were inside the database at any moment, on average.
  Near or above 8 (this instance's vCPUs) means the database is the limit; well below means the searches
  spent their time on the network or in the client.

A short smoke test first checks that worker processes can connect; if they can't, the steps fall back to
threads in this notebook, and the handback says so.

In [ ]:
WORKER_SRC = r"""
import json, os, random, sys, threading, time
from concurrent.futures import ThreadPoolExecutor


def make_connect(cfg):
    if cfg["fixture"]:
        import pg8000.dbapi

        def c():
            return pg8000.dbapi.connect(user="postgres", host=cfg["pghost"], port=cfg["pgport"], database=cfg["db"])
    else:
        from google.cloud.alloydb.connector import Connector, IPTypes
        connector = Connector()

        def c():
            return connector.connect(cfg["instance_uri"], "pg8000", user=cfg["user"], db=cfg["db"],
                                     enable_iam_auth=True, ip_type=IPTypes.PUBLIC)
    return c


def run_worker(cfg):
    connect = make_connect(cfg)
    conns, errs, lock = [], [], threading.Lock()

    def opener(_):
        try:
            c = connect()
            c.autocommit = True
            cur = c.cursor()
            for k, v in cfg["sets"]:
                cur.execute(f"SET {k} = {v}")
            with lock:
                conns.append((c, cur))
        except Exception as e:
            with lock:
                errs.append(f"connect: {type(e).__name__}: {str(e)[:200]}")

    with ThreadPoolExecutor(16) as pool:
        list(pool.map(opener, range(cfg["threads"])))
    with open(cfg["ready"], "w") as f:
        f.write(str(len(conns)))
    deadline = time.time() + 300
    while not os.path.exists(cfg["go"]) and time.time() < deadline:
        time.sleep(0.02)
    start = float(open(cfg["go"]).read()) if os.path.exists(cfg["go"]) else time.time()
    ws, we = start + cfg["warm"], start + cfg["warm"] + cfg["step"]
    per = [[] for _ in conns]

    def loop(i):
        c, cur = conns[i]
        rnd, qids, sql, filtered = random.Random(), cfg["qids"], cfg["sql"], cfg["filtered"]
        while time.time() < start:
            time.sleep(0.002)
        try:
            while True:
                t0 = time.time()
                if t0 >= we:
                    break
                qid = rnd.choice(qids)
                cur.execute(sql, (qid, qid) if filtered else (qid,))
                cur.fetchall()
                t1 = time.time()
                if t0 >= ws and t1 <= we:
                    per[i].append((t1 - t0) * 1000)
        except Exception as e:
            with lock:
                errs.append(f"query: {type(e).__name__}: {str(e)[:200]}")

    ths = [threading.Thread(target=loop, args=(i,)) for i in range(len(conns))]
    for t in ths:
        t.start()
    while time.time() < ws:
        time.sleep(0.01)
    cpu0, w0 = time.process_time(), time.time()
    for t in ths:
        t.join()
    cpu = (time.process_time() - cpu0) / max(time.time() - w0, 1e-6)
    for c, cur in conns:
        try:
            cur.close()
            c.close()
        except Exception:
            pass
    out = {"lat": [x for l in per for x in l], "errs": errs[:10], "n_err": len(errs),
           "connected": len(conns), "cpu": cpu}
    with open(cfg["out"] + ".tmp", "w") as f:
        json.dump(out, f)
    os.replace(cfg["out"] + ".tmp", cfg["out"])
    return out


if __name__ == "__main__":
    run_worker(json.load(open(sys.argv[1])))
    os._exit(0)          # don't wait on the connector's background thread
"""

import itertools
import sys

S = "8 · Load"
WARM = 5
WORKER = BASE / "bench_worker.py"
WORKER.write_text(WORKER_SRC)
RUNDIR = BASE / "bench-run"
RUNDIR.mkdir(exist_ok=True)
_wns = {"__name__": "bench_worker"}
exec(compile(WORKER_SRC, str(WORKER), "exec"), _wns)       # the same code, for the in-notebook fallback
_RUN = itertools.count()
RUN_ID = datetime.now(timezone.utc).strftime("%H%M%S")
for _f in RUNDIR.iterdir():
    _f.unlink()
LOAD = []
MODE = "process"


def pss():
    if not PSS:
        return None
    d = q("SELECT coalesce(sum(calls), 0) AS c, coalesce(sum(total_exec_time), 0) AS t "
          "FROM pg_stat_statements WHERE query LIKE %s", ("SELECT report_id FROM bench%",))
    return float(d["c"][0]), float(d["t"][0])


def launch(label, name, filtered, sets, clients, procs, step_s=None, warm=WARM):
    step_s = step_s or STEP_SECONDS
    k = f"{RUN_ID}-{next(_RUN):02d}"     # unique per notebook run, so files from an earlier run can't interfere
    procs = max(1, min(procs, clients))
    split = [clients // procs + (1 if i < clients % procs else 0) for i in range(procs)]
    go = RUNDIR / f"r{k}.go"
    common = {"fixture": bool(_FIXTURE), "pghost": os.environ.get("CG_PGHOST", "localhost"),
              "pgport": int(os.environ.get("CG_PGPORT", "5432")), "instance_uri": INSTANCE_URI,
              "user": USER, "db": DB_NAME, "qids": QIDS, "sql": search_sql(name, filtered),
              "filtered": filtered, "sets": [list(s) for s in clean(sets)], "warm": warm, "step": step_s,
              "go": str(go)}
    cfgs, handles = [], []
    for i, n in enumerate(split):
        c = dict(common, threads=n, ready=str(RUNDIR / f"r{k}-p{i}.ready"),
                 out=str(RUNDIR / f"r{k}-p{i}.json"))
        (RUNDIR / f"r{k}-p{i}.cfg").write_text(json.dumps(c))
        cfgs.append(c)
    try:
        for i, c in enumerate(cfgs):
            if MODE == "process":
                log = open(RUNDIR / f"r{k}-p{i}.log", "w")
                handles.append(subprocess.Popen([sys.executable, str(WORKER), str(RUNDIR / f"r{k}-p{i}.cfg")],
                                                stdout=log, stderr=subprocess.STDOUT))
            else:
                t = threading.Thread(target=_wns["run_worker"], args=(c,), daemon=True)
                t.start()
                handles.append(t)
        limit = time.time() + 180
        while time.time() < limit and not all(Path(c["ready"]).exists() for c in cfgs):
            if MODE == "process" and all(h.poll() is not None for h in handles):
                break
            time.sleep(0.1)
        s0 = pss()
        start = time.time() + 0.5
        go.with_suffix(".tmp").write_text(repr(start))
        os.replace(go.with_suffix(".tmp"), go)
        for h in handles:
            if MODE == "process":
                try:
                    h.wait(timeout=warm + step_s + 180)
                except subprocess.TimeoutExpired:
                    h.kill()
            else:
                h.join(warm + step_s + 180)
        end = time.time()
        s1 = pss()
    finally:
        for h in handles:
            if MODE == "process" and h.poll() is None:
                h.kill()
    lat, errs, cpus, conn = [], [], [], 0
    for i, c in enumerate(cfgs):
        o = Path(c["out"])
        if o.exists():
            d = json.loads(o.read_text())
            lat += d["lat"]; errs += d["errs"]; cpus.append(d["cpu"]); conn += d["connected"]
        else:
            lg = RUNDIR / f"r{k}-p{i}.log"
            errs.append("worker produced no result: " + (lg.read_text()[-300:] if lg.exists() else "no log"))
    lat.sort()
    pct = (lambda p: lat[min(len(lat) - 1, int(p * len(lat)))] if lat else float("nan"))
    row = {"label": label, "index": name, "filtered": filtered, "clients": clients, "procs": procs,
           "connected": conn, "qps": len(lat) / step_s, "p50": pct(0.5), "p95": pct(0.95), "p99": pct(0.99),
           "errors": len(errs), "client_cpu_max": max(cpus) if cpus else None,
           "load_start": start, "load_end": end, "mode": MODE, "first_error": errs[0] if errs else ""}
    if s0 and s1 and s1[0] > s0[0]:
        dur = end - start
        row["server_ms"] = (s1[1] - s0[1]) / (s1[0] - s0[0])
        row["busy_in_server"] = (s1[1] - s0[1]) / 1000 / dur
        row["server_qps"] = (s1[0] - s0[0]) / dur
    return row


def show(row):
    sv = (f"  server {row['server_ms']:.1f} ms/search, busy in server {row['busy_in_server']:.1f}"
          if "server_ms" in row else "")
    line = (f"{row['label']:<58} {row['qps']:>6.0f} q/s  p50 {row['p50']:6.1f}  p95 {row['p95']:6.1f}  "
            f"p99 {row['p99']:6.1f} ms  client CPU max {row['client_cpu_max'] or 0:.0%} "
            f"({row['procs']} proc){sv}  errors {row['errors']}")
    print(line)
    hb(S, line + (f"—{row['first_error']}" if row["errors"] else ""))


# ---- smoke test: can separate processes connect? -------------------------------------------
smoke = launch("smoke test", "ScaNN", False, SCANN_TUNED + FORCE, clients=2, procs=1, step_s=2, warm=0)
if smoke["qps"] == 0:
    MODE = "thread"
    line = f"worker processes failed ({smoke['first_error'][:200]}); falling back to threads in this notebook"
else:
    line = f"worker processes connect: {smoke['connected']} connections, {smoke['qps']:.0f} q/s in the smoke test"
print(line + "\n")
hb(S, line)

LOADS = [
    ("plain: ScaNN 03 settings, index forced", "ScaNN", False, SCANN_BASE + FORCE, CLIENT_PROCS),
    ("plain: ScaNN tuned, index forced", "ScaNN", False, SCANN_TUNED + FORCE, CLIENT_PROCS),
    (f"plain: HNSW {E}, index forced", "HNSW", False, HNSW_SET + FORCE, CLIENT_PROCS),
    (f"plain: HNSW {E}, index forced, ONE client process", "HNSW", False, HNSW_SET + FORCE, 1),
    ("filtered: exact (no vector index)", "Exact", True, [], CLIENT_PROCS),
    ("filtered: ScaNN filtered setting, index forced", "ScaNN", True, SCANN_FTUNED + FORCE, CLIENT_PROCS),
    (f"filtered: HNSW {E}, index forced, iterative scan", "HNSW", True, HNSW_SET + [ITER] + FORCE, CLIENT_PROCS),
]
if not RUN_LOAD:
    print("skipped—RUN_LOAD is False")
else:
    hb(S, f"{LOAD_CLIENTS} clients per step; {STEP_SECONDS}s measured after {WARM}s warm-up; mode {MODE}")
    for label, name, filtered, sets, procs in LOADS:
        if ITER in sets and KNOB.get("hnsw.iterative_scan") is None:
            continue
        row = launch(label, name, filtered, sets, LOAD_CLIENTS, procs)
        LOAD.append(row)
        show(row)

---
## 9 · Instance CPU, summary and handback

AlloyDB reports CPU to Cloud Monitoring once a minute, a few minutes late, so this cell waits (up to
five minutes) until the last load step is covered, then lists CPU minute by minute next to the step that
was running. The handback box follows.

In [ ]:
import html as _html
from IPython.display import HTML, display

S = "9 · Instance CPU"


def _iso(t):
    return datetime.fromtimestamp(t, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def cpu_points(t0, t1):
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    cr, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    s = AuthorizedSession(cr)
    base = f"https://monitoring.googleapis.com/v3/projects/{PROJECT}"
    r = s.get(f"{base}/metricDescriptors",
              params={"filter": 'metric.type = starts_with("alloydb.googleapis.com/instance/cpu")'})
    types = [d["type"] for d in r.json().get("metricDescriptors", [])] if r.ok else []
    pick = (next((t for t in types if t.endswith("average_utilization")), None)
            or next((t for t in types if "utilization" in t), None))
    if not pick:
        return None, [], f"no AlloyDB CPU metric visible ({r.status_code}: {types or r.text[:200]})"
    for flt in (f'metric.type = "{pick}" AND resource.labels.instance_id = "{INSTANCE}"', f'metric.type = "{pick}"'):
        r = s.get(f"{base}/timeSeries", params={"filter": flt, "interval.startTime": _iso(t0),
                                                 "interval.endTime": _iso(t1)})
        series = r.json().get("timeSeries", []) if r.ok else []
        if series:
            break
    pts = {}
    for ts in series:
        for p in ts.get("points", []):
            v = p["value"].get("doubleValue", p["value"].get("int64Value"))
            pts.setdefault(pd.Timestamp(p["interval"]["endTime"]).timestamp(), []).append(float(v))
    return pick, sorted((t, sum(v) / len(v)) for t, v in pts.items()), ("" if r.ok else r.text[:200])


TIMELINE = []
if LOAD and not _FIXTURE:
    t_first, t_last = LOAD[0]["load_start"] - 120, LOAD[-1]["load_end"]
    deadline = time.time() + 300
    while True:
        try:
            metric, pts, err = cpu_points(t_first, max(time.time(), t_last + 60))
        except Exception as e:  # noqa: BLE001
            metric, pts, err = None, [], f"{type(e).__name__}: {str(e)[:200]}"
        if (pts and pts[-1][0] >= t_last) or time.time() > deadline or (metric is None and err):
            break
        print(f"waiting for Cloud Monitoring ({len(pts)} points so far, latest "
              f"{datetime.fromtimestamp(pts[-1][0]).strftime('%H:%M:%S') if pts else '—'})")
        time.sleep(30)
    scale = 100 if pts and max(v for _, v in pts) <= 1.0 else 1
    for t, v in pts:
        best, ov = "", 0
        for r in LOAD:
            o = min(t, r["load_end"]) - max(t - 60, r["load_start"])
            if o > ov:
                best, ov = r["label"], o
        TIMELINE.append((t, v * scale, best, ov))
        line = (f"{datetime.fromtimestamp(t, timezone.utc).strftime('%H:%M:%S')}Z  CPU {v * scale:5.1f}%  "
                + (f"← {best} ({ov:.0f}s of the minute)" if best else "(between steps)"))
        print(line)
        hb(S, line)
    if not pts:
        print(f"no CPU data: {err}")
        hb(S, f"no CPU data ({metric}): {err}")
    else:
        hb(S, f"metric {metric}")
else:
    hb(S, "skipped (offline test)" if _FIXTURE else "skipped (no load steps)")

if LOAD:
    L = pd.DataFrame(LOAD)
    cols = [c for c in ["label", "qps", "p50", "p95", "p99", "client_cpu_max", "server_ms", "busy_in_server",
                        "errors"] if c in L]
    print("\n" + L[cols].round(2).to_string(index=False))

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = {"run_utc": stamp, "project": PROJECT, "shape": SHAPE, "runtime_vcpu": os.cpu_count(), "versions": VER,
       "rows": N_ROWS, "settings": GUCS, "sizes_mb": SIZES, "scann_sweep": SC, "scann_tuned": clean(SCANN_TUNED),
       "hnsw_sweep": HN, "hnsw_chosen": HNSW_EF_CHOSEN, "hnsw_cliff": cliff,
       "scann_filtered_sweep": FSC, "scann_filtered_setting": clean(SCANN_FTUNED),
       "plans": {k: {kk: vv for kk, vv in v.items()} for k, v in PLANS.items()},
       "load": LOAD, "cpu_timeline": TIMELINE, "target_recall": TARGET_RECALL}
p = BASE / f"index-diagnostics-{stamp}.json"
p.write_text(json.dumps(OUT, indent=2, default=str))
r = gcs("cp", str(p), f"{RESULTS_PREFIX}/{p.name}")
saved = r.returncode == 0
print(f"\nresults {'saved to ' + RESULTS_PREFIX + '/' + p.name if saved else 'NOT saved to the bucket: ' + r.stderr[:200]}")

lines = ["=" * 70, "CYMBALGOAL INDEX DIAGNOSTICS—04 HANDBACK", f"run {stamp}", "=" * 70]
order = [k for k in HB if k != "7b · Plan text"] + (["7b · Plan text"] if "7b · Plan text" in HB else [])
for sec in order:
    lines += ["", f"## {sec}", *HB[sec]]
writes = [c for c in GCS_LOG if c.split()[2] == "cp"]
bad = [c for c in writes if not any(_under(c.split()[-1], w) for w in WRITE_OK)]
lines += ["", "## Bucket audit",
          f"calls {len(GCS_LOG)}; writes {len(writes)}, all allowed: {not bad}; results saved: {saved}", "=" * 70]
TEXT = "\n".join(lines)
(BASE / "handback-04.txt").write_text(TEXT)
assert not bad, f"a write left the allowed prefixes: {bad}"

display(HTML(f'''
<div style="font-family:system-ui,sans-serif;max-width:100%">
  <button id="cg-copy" style="font-size:15px;font-weight:700;padding:10px 20px;border-radius:8px;border:0;
      background:#1f8a6d;color:#fff;cursor:pointer">Copy handback</button>
  <span id="cg-msg" style="margin-left:12px;font-weight:600;color:#1f8a6d"></span>
  <textarea id="cg-box" readonly style="display:block;width:100%;height:340px;margin-top:10px;
      font:12px/1.4 ui-monospace,Menlo,monospace;padding:10px;box-sizing:border-box">{_html.escape(TEXT)}</textarea>
</div>
<script>
(function() {{
  const b = document.getElementById("cg-copy"), t = document.getElementById("cg-box"), m = document.getElementById("cg-msg");
  b.addEventListener("click", async () => {{
    try {{ await navigator.clipboard.writeText(t.value); m.textContent = "Copied—paste it into the chat."; }}
    catch (e) {{ t.focus(); t.select(); m.textContent = "Clipboard blocked: the text is selected, use Edit → Copy."; }}
  }});
}})();
</script>'''))
print(f"{len(TEXT):,} characters; also saved to {BASE / 'handback-04.txt'}")